# Calling Bedrock Models with LangChain

Demonstrates how to build AI applications by calling Amazon Bedrock foundation models through LangChain. Includes a working staff scheduling chatbot that accepts natural language requests and routes them through a Bedrock-backed LangChain chain.

**Key concepts covered:**
- Configuring `ChatBedrock` as a LangChain LLM backend
- Building prompt templates and chains with LangChain Expression Language (LCEL)
- Synchronous vs. streaming invocation
- Practical staff scheduling pattern

## Setup

In [ ]:
%pip install -q langchain langchain-aws boto3

In [ ]:
import boto3
from langchain_aws import ChatBedrock
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

# Verify AWS credentials are configured
session = boto3.session.Session()
print('AWS region:', session.region_name)
print('AWS identity:', boto3.client('sts').get_caller_identity()['Arn'])

## 1. Configure ChatBedrock

`ChatBedrock` is the LangChain wrapper around Amazon Bedrock. It accepts the same `model_id` strings used in the Bedrock console and SDK.

In [ ]:
llm = ChatBedrock(
    model_id='anthropic.claude-3-sonnet-20240229-v1:0',
    region_name=session.region_name,
    model_kwargs={
        'max_tokens': 1024,
        'temperature': 0.3,
    }
)

print('Model configured:', llm.model_id)

## 2. Direct Invocation

The simplest pattern: pass a list of messages directly to the model.

In [ ]:
messages = [
    SystemMessage(content='You are a concise AI assistant. Answer in one sentence.'),
    HumanMessage(content='What is Amazon Bedrock?')
]

response = llm.invoke(messages)
print(response.content)

## 3. Prompt Templates and LCEL Chains

LangChain Expression Language (LCEL) composes steps with the `|` operator: `prompt | llm | parser`.

In [ ]:
# Build a simple Q&A chain
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Be concise and direct.'),
    ('human',  '{question}')
])

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({'question': 'What are the main Amazon Bedrock foundation model families?'})
print(answer)

## 4. Streaming Invocation

Streaming returns tokens as they are generated — useful for responsive UIs.

In [ ]:
streaming_llm = ChatBedrock(
    model_id='anthropic.claude-3-sonnet-20240229-v1:0',
    region_name=session.region_name,
    streaming=True,
    model_kwargs={'max_tokens': 512, 'temperature': 0.3}
)

stream_chain = prompt | streaming_llm | StrOutputParser()

print('Streaming response:')
for chunk in stream_chain.stream({'question': 'Explain LangChain Expression Language in two sentences.'}):
    print(chunk, end='', flush=True)
print()  # newline after streaming completes

## 5. Staff Scheduling Chatbot

A practical example: a chatbot that accepts natural language scheduling requests and returns structured shift assignments.

The chain uses a system prompt with scheduling context and parses a JSON response from the model.

In [ ]:
import json
import re

STAFF = [
    {'name': 'Alice',   'role': 'barista',    'max_hours': 40},
    {'name': 'Bob',     'role': 'barista',    'max_hours': 24},
    {'name': 'Carlos',  'role': 'supervisor', 'max_hours': 40},
    {'name': 'Diana',   'role': 'barista',    'max_hours': 32},
    {'name': 'Evan',    'role': 'supervisor', 'max_hours': 40},
]

scheduling_prompt = ChatPromptTemplate.from_messages([
    ('system', """
You are a staff scheduling assistant for a coffee shop.
Available staff: {staff_list}

Rules:
- Every shift must have at least one supervisor
- No staff member can exceed their max_hours per week
- Shifts are 8 hours unless otherwise specified

Return ONLY valid JSON in this format (no commentary):
{{
  "schedule": [
    {{"day": "Monday", "shift": "morning", "staff": ["name1", "name2"]}}
  ],
  "notes": "any important notes"
}}
"""),
    ('human', '{request}')
])

scheduling_chain = scheduling_prompt | llm | StrOutputParser()

staff_list = json.dumps(STAFF, indent=2)
request    = 'Schedule the morning shift (6am-2pm) for Monday through Wednesday. Make sure every shift is covered.'

raw_output = scheduling_chain.invoke({'staff_list': staff_list, 'request': request})
print('Raw model output:')
print(raw_output)

In [ ]:
# Parse the JSON response
def extract_json(text):
    """Extract JSON block from model output, handling markdown code fences."""
    match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if match:
        return json.loads(match.group(1))
    return json.loads(text.strip())

schedule = extract_json(raw_output)

print('=== Generated Schedule ===')
for shift in schedule.get('schedule', []):
    print(f"{shift['day']} {shift['shift']:10s} → {', '.join(shift['staff'])}")

if schedule.get('notes'):
    print('\nNotes:', schedule['notes'])

## 6. Multi-Turn Conversation with Memory

Extend the scheduling chatbot to handle follow-up requests by passing the conversation history.

In [ ]:
from langchain_core.messages import AIMessage

conversation_history = [
    SystemMessage(content=f'You are a staff scheduling assistant. Available staff: {staff_list}. Be concise.')
]

def chat(user_message: str) -> str:
    conversation_history.append(HumanMessage(content=user_message))
    response = llm.invoke(conversation_history)
    conversation_history.append(AIMessage(content=response.content))
    return response.content

# Turn 1
print('User: Who is available for the Thursday evening shift?')
print('Assistant:', chat('Who is available for the Thursday evening shift?'))
print()

# Turn 2 — follow-up referencing prior context
print('User: Can Alice cover that shift if Carlos calls in sick?')
print('Assistant:', chat('Can Alice cover that shift if Carlos calls in sick?'))

## Summary

| Pattern | When to Use |
|---------|-------------|
| `llm.invoke(messages)` | One-off calls, debugging |
| `prompt \| llm \| parser` (LCEL) | Reusable chains with templated inputs |
| `streaming=True` | Responsive UIs needing incremental output |
| Conversation history list | Multi-turn chatbots with context retention |

The `ChatBedrock` wrapper makes it straightforward to swap in any Bedrock model (Claude, Titan, Nova, etc.) by changing the `model_id` — the rest of the chain stays the same.